# Fraud DNA Prototype

This notebook performs EDA, a prototype Fraud DNA score, trains LightGBM/XGBoost ensemble, computes PR-AUC, and generates a SHAP explanation for a top sample.
Save models to `../models` and outputs to `../outputs`.

In [ ]:
# Section 1: Load libraries and dataset

import os
from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = os.path.join('..','DataSet.csv')
print('Data path:', DATA_PATH)
df = pd.read_csv(DATA_PATH)
print('Rows,Cols:', df.shape)
display(Markdown('**Target distribution (F3924) if present:**'))
if 'F3924' in df.columns:
    display(df['F3924'].value_counts())
else:
    display(Markdown('F3924 not found'))

In [ ]:
# Section 2: Parameters and feature selection
anchors = ['F321','F3836','F2082']
features = ['F115','F321','F527','F531','F670','F1692','F2082','F2122','F2582','F2678','F2737','F2956','F3043','F3836','F3887','F3889','F3891','F3894']
available = [f for f in features if f in df.columns]
display(Markdown('**Available features:**'))
display(available)
OUT_DIR = os.path.join('..','models')
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Section 3: EDA for anchor features
for a in anchors:
    display(Markdown(f'**Feature: {a}**'))
    if a in df.columns:
        miss = df[a].isna().mean()
        display(Markdown(f'- missing: {miss:.2%}'))
        plt.figure(figsize=(6,2))
        df[a].dropna().astype(float).hist(bins=40)
        plt.title(a)
        plt.show()
    else:
        display(Markdown('- not present'))

In [ ]:
# Section 4: Compute prototype Fraud DNA score
if 'F3924' in df.columns and all(a in df.columns for a in anchors):
    positives = df[df['F3924']==1]
    if len(positives)>0:
        canon = positives[anchors].astype(float).mean().values
        def frauddna_score_row(row):
            vec = row[anchors].astype(float).values
            dist = np.linalg.norm(vec - canon)
            return 1.0 / (1.0 + dist)
        df['frauddna_score'] = df.apply(lambda r: frauddna_score_row(r) if not r[anchors].isnull().any() else 0.0, axis=1)
        display(df['frauddna_score'].describe())
    else:
        display(Markdown('No positives to build canonical Fraud DNA'))
else:
    display(Markdown('Missing anchors or target; skipping Fraud DNA score'))

In [ ]:
# Section 5: Train ensemble models
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, auc
import lightgbm as lgb
import xgboost as xgb
import joblib

if 'F3924' in df.columns:
    X = df[available].fillna(0).astype(float)
    if 'frauddna_score' in df.columns:
        X['frauddna_score'] = df['frauddna_score']
    y = df['F3924']
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    # LightGBM
    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)
    params = {'objective':'binary','metric':'auc','verbosity':-1}
    gbm = lgb.train(params, lgb_train, num_boost_round=200, valid_sets=[lgb_val], early_stopping_rounds=10, verbose_eval=False)
    # XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    xparams = {'objective':'binary:logistic','eval_metric':'auc'}
    bst = xgb.train(xparams, dtrain, num_boost_round=200, evals=[(dval,'val')], early_stopping_rounds=10, verbose_eval=False)
    # Ensemble
    pred_lgb = gbm.predict(X_val, num_iteration=gbm.best_iteration)
    pred_xgb = bst.predict(xgb.DMatrix(X_val))
    ensemble = 0.5*pred_lgb + 0.5*pred_xgb
    precision, recall, _ = precision_recall_curve(y_val, ensemble)
    pr_auc = auc(recall, precision)
    display(Markdown(f'**Validation PR AUC:** {pr_auc:.4f}'))
    # Save models and snapshot
    joblib.dump(gbm, os.path.join(OUT_DIR,'lgb_model.joblib'))
    bst.save_model(os.path.join(OUT_DIR,'xgb_model.json'))
    os.makedirs(os.path.join('..','outputs'), exist_ok=True)
    df_snapshot = X_val.copy()
    df_snapshot['ensemble_prob'] = ensemble
    df_snapshot['label'] = y_val.values
    df_snapshot.to_csv(os.path.join('..','outputs','cti_snapshot.csv'), index=False)
    display(Markdown('Saved models and CTI snapshot.'))
else:
    display(Markdown('Target F3924 missing; cannot train.'))

In [ ]:
# Section 6: SHAP explainability (LightGBM)
try:
    import shap
    shap_values = None
    explainer = shap.TreeExplainer(gbm)
    shap_values = explainer.shap_values(X_val)
    # pick top ensemble index
    idx = int(np.argmax(ensemble))
    plt.figure(figsize=(8,4))
    shap.summary_plot(shap_values, X_val, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join('..','outputs','shap_summary.png'))
    display(Markdown('Saved SHAP summary to ../outputs/shap_summary.png'))
    # save shap for sample
    sample_shap = {'expected_value': explainer.expected_value, 'shap_values': shap_values[idx], 'features': list(X_val.columns)}
    joblib.dump(sample_shap, os.path.join('..','outputs','shap_sample.joblib'))
except Exception as e:
    display(Markdown(f'Could not compute SHAP: {e}'))

# Section 7: Notes and next steps
- This notebook is a prototype. Next steps: implement true multivariate DTW trajectories, ring detection using NetworkX, cohort-relative scoring, and production SHAP explainers as a service.
